In [1]:
# The package named "fitz" on PyPI is NOT PyMuPDF and can trigger:
# "ModuleNotFoundError: No module named 'frontend'".
# PyMuPDF is installed from the "pymupdf" package, but imported as "fitz".

#%pip uninstall -y fitz frontend
%pip install -U pymupdf

import re
import pymupdf
import pandas as pd


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

PDF_PATH = "data_raw/regulation_docs/FIA 2026 F1 Regulations - Section A [General Regulatory Provisions] - Iss 01 - 2025-12-10.pdf"  # or pymupdf.Document(filename)
MODALS = r"\b(must|shall|may|must not|shall not|prohibited|not permitted|required)\b" # modals to identify obligations and permissions
MEASURE = r"(\d+(?:\.\d+)?)\s?(mm|cm|m|kg|g|N|kN|%)\b" # measurement patterns (e.g., "100 mm", "2.5 kg", "50%")
CONTENTS = 59  # number of pages in the table of contents (adjust for specific doc)

In [3]:
def extract_pages(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path) # open doc
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({"page": i + 1, "text": text}) # store page number and text in a list of dicts
    return pages

pages = extract_pages(PDF_PATH)
df = pd.DataFrame(pages)
print(pages[0]["text"][:500])  # Print the first 500 characters of the first page's text col

SECTION A: GENERAL REGULATORY PROVISIONS 
 
 
 
0  A 
A1 
2026 Formula 1 Regulations: General Regulatory Provisions 
©2025 Fédération Internationale de l’Automobile 
10 December 2025
Issue 01
SECTION A: GENERAL REGULATORY PROVISIONS 
 
Version: 
 
 
Issue 01 
Status:  
 
 
PUBLISHED 
Date:  
 
 
10/12/2025 
WMSC approval date:  
10/12/2025 
 
CONVENTION: 
Black Text: 
Regulations approved by the WMSC on 10/12/2025 
[Red Text]: 
Information on applicable Governance and relevant Advisory Committee


In [4]:
def clean_text(t: str) -> str:
    # fix hyphenated line breaks: "inter-\nnal" -> "internal"
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)
    # join lines that are broken mid-sentence
    t = re.sub(r"\n+", "\n", t)
    # normalise spaces
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

In [5]:
def split_into_clauses(text: str) -> list[dict]:
    """
    Example clause pattern: A1.2, A3.1.2, B2.4 etc.
    Supports an optional letter prefix before the clause number.
    """
    # Updated regex: optional letter prefix (e.g. "A"), then digits separated by dots
    pattern = re.compile(r"(?m)^(?P<id>[A-Z]?\d+(?:\.\d+)+)\s+")
    # Find ALL clause headings (e.g. "3.1", "3.1.2") in the text and collect them as a list
    matches = list(pattern.finditer(text))
    clauses = []

    # If no clause headings were found, return the entire text as a single
    # "unidentified" clause with clause_id set to None
    if not matches:
        return [{"clause_id": None, "clause_text": text.strip()}]

    # Loop through each matched clause heading
    for idx, m in enumerate(matches):
        # The current clause starts where this heading match begins
        start = m.start()

        # The current clause ends where the NEXT heading begins,
        # or at the end of the full text if this is the last clause
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)

        # Extract the clause number (e.g. "3.1.2") from the named capture group "id"
        clause_id = m.group("id")

        # Slice out everything from this heading to the next heading — that's the full clause text
        clause_text = text[start:end].strip()

        # Store the clause as a dict with its id and text
        clauses.append({"clause_id": clause_id, "clause_text": clause_text})
    return clauses

the code splits a page of regulation text into individual clauses by finding numbered headings (like 3.1, 3.1.2). Each clause's text runs from its heading up to the start of the next heading (or the end of the page). If no headings are found, the whole text is returned as one clause.

In [ ]:
def enrich_clause(clause_text: str) -> dict:
    # Search the clause text for modal verbs (e.g. "must", "shall", "may", "prohibited")
    # that indicate regulatory obligations or permissions
    modals = re.findall(MODALS, clause_text, flags=re.IGNORECASE)

    # Search for numeric measurements with units (e.g. "100 mm", "2.5 kg", "50%")
    # Each match returns a tuple of (value, unit)
    measures = re.findall(MEASURE, clause_text)

    return {
        # True if any modal/constraint language was found in the clause
        "has_constraint_language": bool(modals),
        # Deduplicated, sorted list of modal terms found (all lowercased)
        "modal_terms": sorted(set([m.lower() for m in modals])),
        # List of dicts, each containing a numeric value and its unit
        "measurements": [{"value": v, "unit": u} for v, u in measures],
    }

In [ ]:
def build_dataset(pdf_path: str) -> pd.DataFrame:
    # Extract raw text from every page of the PDF
    pages = extract_pages(pdf_path)
    rows = []

    # Loop through each page's dict (containing "page" number and "text")
    for p in pages:
        # Clean up the raw text (fix hyphenation, normalise whitespace, etc.)
        cleaned = clean_text(p["text"])

        # Split the cleaned page text into individual clauses (e.g. A1.2, A3.1.2)
        for clause in split_into_clauses(cleaned):
            # Enrich each clause with modal terms and measurements
            meta = enrich_clause(clause["clause_text"])

            # Combine page number, clause id, clause text, and enriched metadata
            # into a single flat dict. The ** unpacks the meta dict so its keys
            # (has_constraint_language, modal_terms, measurements) become top-level entries
            rows.append({
                "page": p["page"],
                "clause_id": clause["clause_id"],
                "text": clause["clause_text"],
                **meta
            })

    # Convert the list of row dicts into a pandas DataFrame — one row per clause
    return pd.DataFrame(rows)

In [8]:
df = build_dataset(PDF_PATH)
df = df.iloc[CONTENTS:].reset_index(drop=True)  # drop first 58 rows (table of contents)
print(df)

     page clause_id                                               text  \
0       4      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
1       5      A1.1                                    A1.1 \nOverview   
2       5    A1.1.1  A1.1.1 \nThe FIA is responsible for the sporti...   
3       5    A1.1.2  A1.1.2 \nThe Championship is the exclusive pro...   
4       5    A1.1.3  A1.1.3 \nThis Section A (General Regulatory Pr...   
..    ...       ...                                                ...   
297    82       5.5  5.5 \nNo right of appeal \nPU Manufacturers sh...   
298    82       6.1  6.1 \nAn Automotive Manufacturer is a Manufact...   
299    82       6.2  6.2 \nThe Core Activities of an Automotive Man...   
300    83      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   
301    84      None  SECTION A: GENERAL REGULATORY PROVISIONS \n \n...   

     has_constraint_language         modal_terms measurements  
0                      False                  [